# Universal Human Tracking with YOLO11
File này được chuẩn bị chuyên nghiệp để huấn luyện mô hình nhận diện người sử dụng kiến trúc **YOLO11** - phiên bản mới nhất và tối ưu nhất của Ultralytics.

Để hạn chế tối đa lỗi nhảy ID (ID switching), việc sử dụng một mô hình lớn hơn có nhiều tham số (ví dụ: **Medium** hoặc **Large**) sẽ cải thiện độ chính xác đáng kể, giảm thiểu việc bỏ sót đối tượng (False Negatives) - nguyên nhân trực tiếp dẫn đến nhảy ID.

In [ ]:
# 1. Môi trường và Thư viện
!nvidia-smi
%pip install ultralytics roboflow

In [ ]:
# 2. Tải Dataset từ Roboflow (Đã cập nhật lên Phiên bản 2)
from roboflow import Roboflow
rf = Roboflow(api_key="le73LYaS0c1EiJwTNssZ")
project = rf.workspace("facedetection-uqkmv").project("human_26")
dataset = project.version(2).download("yolov8") # Đổi từ version(1) sang version(2) vì version(2) đã được sinh dữ liệu thành công

In [ ]:
# 3. Huấn luyện (Training) với YOLO11 - Chọn mô hình nhiều tham số hơn
from ultralytics import YOLO

# DANH SÁCH MÔ HÌNH YOLO11 CÓ THỂ CHỌN:
# - 'yolo11n.pt' : Nano  (2.6M params) - Nhanh nhất, nhẹ nhất, nhưng độ chính xác thấp (dễ nhảy ID).
# - 'yolo11s.pt' : Small (9.4M params) - Nhanh, độ chính xác khá tốt.
# - 'yolo11m.pt' : Medium (20.1M params) - [KHUYÊN DÙNG]: Cân bằng hoàn hảo, độ chính xác cao, bám vết cực tốt.
# - 'yolo11l.pt' : Large (25.3M params) - Độ chính xác rất cao, tối ưu cho GPU rời.
# - 'yolo11x.pt' : X-Large (56.9M params) - Lớn nhất, chính xác cao nhất nhưng chạy chậm nhất.

MODEL_SIZE = 'yolo11m.pt'  # Hãy thay đổi model tại đây (ví dụ: yolo11m.pt hoặc yolo11l.pt)

print(f"--- Khởi tạo huấn luyện với mô hình lớn hơn: {MODEL_SIZE} ---")
model = YOLO(MODEL_SIZE) 

results = model.train(
    data=f"{dataset.location}/data.yaml", 
    epochs=50,             # Bạn có thể tăng lên 100 epochs nếu muốn model học kỹ hơn
    imgsz=640, 
    device=0,              # Sử dụng GPU đầu tiên của Colab
    patience=15,           # Dừng sớm nếu không cải thiện sau 15 epoch
    save=True
)

In [ ]:
# 4. Đánh giá và Tải Model tốt nhất về máy
from google.colab import files
import os

best_pt = 'runs/detect/train/weights/best.pt'
if os.path.exists(best_pt):
    files.download(best_pt)
else:
    print("Hệ thống không tìm thấy file trọng số. Vui lòng kiểm tra lại logs huấn luyện.")